# Example of using C++ code in python and Jupyter with cppyy

This notebook demonstrates how to use C++ code in python using [cppyy](https://cppyy.readthedocs.io). cppyy works quite differently from pybind11/nanobind or SWIG: it generates the python bindings at *runtime*, i.e., it compiles and loads the C++ code on demand, when you run the python code. (It does technically have the ability to load pre-compiled code as well, but at that point, you might as well use pybind11, the industry standard.) This is super convenient for "quick and dirty" applications, like, for example, CompPhys homework assignments :)

It should be mentioned that cppyy is a small project maintained by a small team (github.com/compiler-research). It is primarilly used in High Energy Physics, where it underlies the "PyROOT" package. The original cppyy project was recently archived and replaced by [cppjit](https://github.com/compiler-research/cppjit). So, it's a bit less stable than the other options; keep this in mind before you build your whole house on it :)

<div>
<img src="nickcage.webp" width="500" alt="Nick Cage's house falling into the Pacific Ocean"/>
<p>Nick Cage's house falling into the Pacific Ocean</p>
</div>


## Step 1: Look at the C++ files

These are **exactly the same two files** used in the SWIG and pybind11 examples — nothing about the underlying C++ changes when you switch binding tools.


In [ ]:
! echo ".h:"
! cat cppyy_example/example.h 
! echo "\ncpp:"
! cat cppyy_example/example.cpp


## Step 2: ...that's it?

With SWIG, this is where you'd write a `.i` interface file. With pybind11, you need a `binding.cpp` file. With cppyy, there's nothing to write and nothing to compile — you just tell it which header (and, since we haven't built a separate library, which source file) to look at:


In [ ]:
import cppyy

cppyy.include("cppyy_example/example.h")
cppyy.include("cppyy_example/example.cpp");  # semicolon just suppresses the notebook printing cppyy's return value


The first time you run this in a fresh environment, cppyy has to build and cache a precompiled header for its own standard-library headers, so it can take a little while and print some build chatter. After that, it's just a normal (if slow-ish) import.

Now the function is available straight from `cppyy.gbl`, the namespace that mirrors the global C++ namespace:


In [ ]:
from cppyy.gbl import sum_int

x = sum_int([1, 2, 3])
print(x)


A plain python list converts to `std::vector<int>` automatically — no `%template` line (SWIG) and no `#include <pybind11/stl.h>` (pybind11) required; cppyy figures out the conversion from the function signature it just parsed.

Note: I did not measure the overhead of converting a plain python list to a `std::vector<int>`. This is almost certainly a time-consuming step: a python list is not contiguous in memory, while a `std::vector<int>` is continuous. 


# An even simpler example

In [ ]:
cppyy.cppdef("""
#include <iostream>
void hello() {
    std::cout << "Hello!" << std::endl;
}
""")


In [ ]:
from cppyy.gbl import hello
hello()


## Recap

|  | SWIG | pybind11 | cppyy |
|---|---|---|---|
| Interface/binding code you write | `.i` file | `binding.cpp` | none |
| Build step | Yes (codegen + compile) | Yes (compile) | No (JIT, at import time) |
| Ships as | a compiled `.so` you can distribute | a compiled `.so` you can distribute | nothing to distribute — needs cppyy + your source every time |

Fine for a one-off script or exploring someone else's headers interactively; not what you'd reach for to package and ship something.
